# 🎯 [QLT04_M09] AAR 기반 가동률 예측치와 실제치 편차 분석 및 분석 파라미터 재보정 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/QLT04/qlt04_m09_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 품질·규격·검사 (Level 2) |
| **미션 ID** | QLT04_M09 |
| **미션 주제** | AAR 기반 가동률 예측치와 실제치 편차 분석 및 분석 파라미터 재보정 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
외주정비는 정비 대기, 외주 검사 대기, 재작업, 규격·제원 불일치가 겹치면서 장비 가동률을 떨어뜨리지만 요인별 영향이 한눈에 보이지 않아 어디에 정비 자원을 먼저 투입할지 정하기 어렵다. 본 프로젝트는 미 국방부 외주정비 품질보증 사례의 문제의식(규격 정합성·검사 병목·하자 연결)을 교육용 가상 자료로 재구성해, 정비 대기 요인 분류, 저가동 요인 분석, 재배치 시뮬레이션, 추세 대시보드, 사람 검토 워크플로우까지 프로토타입으로 구현하는 과제이다. 가상 장비는 유형 A·B·C와 가상 부대로만 구분하며 실제 장비·부대 자료는 쓰지 않는다. 최종 목표는 가동률 저하 요인과 외주 품질검사 병목을 근거와 함께 보여 주되, 정비 우선순위와 자원 투입 결정은 정비관·품질검사관·군수참모가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 프로세스 정의서(`qlt04_as_is_to_be_spec.md`), UAT 피드백 리포트(`qlt04_uat_feedback_report.md`), 운영 매뉴얼(`qlt04_final_manual.md`) 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 품질 판정 룰 엔진, 릿지 회귀 XAI 엔진, 시뮬레이터 UI, HITL 결재 FSM, 배치 러너 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 야전 배포를 위한 최종 패키지(`qlt04_package/`)

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 09. AAR 기반 가동률 예측치와 실제치 편차 분석 및 분석 파라미터 재보정
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

AAR 기반 가동률 예측치와 실제치 편차를 분석하고 파라미터를 재보정한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **AAR로 가동률 예측치와 실제치의 편차를 분석해 원인을 규명하고, 분석 파라미터를 재보정해 정확도 개선 효과와 UAT 점검 결과를 확인한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **평시 자료로 학습한 모델의 예측을 외주 검사 병목이 악화된 새 분기 실측과 부대·유형 셀 단위로 비교해 편향과 원인을 요인별로 분해 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/qlt04_uat_feedback_report.md 및 패치 내역` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 실무 업무 생산성 기여도 및 현장 적응성

#### 미션 PASS 조건

- 정량 기준: UAT 체크리스트 만족도 90점 이상, 개선 요구 5건 반영 완료


- 진행 기준: UAT 결과 보고서 작성 및 최종 버그 픽스 완료


- 검증 방법: 실제 업무 시나리오 기반 최종 검수 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""가동률 예측치와 실제치의 편차를 부대·유형 셀 단위로 AAR 분석해 오차 원인(외주 검사 병목 악화)을 요인별로 분해하고, 편향 보정·병목 변수 추가·가중 재학습·정규화 강도 조정·예측 구간의 5개 재보정 패치를 단계별로 검증해 채택 여부를 정하며 UAT 체크리스트와 재보정 파라미터(YAML 문자열)를 출력한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 장비 가동률·정비 대기 자료 생성기 (실제 부대·장비·제원·가동률 정보 없음)
# 장비 유형 A(궤도형)·B(화포형)·C(차륜형)은 추상 구분이며 실제 장비와 무관하다.
# 최근 90일 동안 장비 1대의 정비 대기 요인(부품·정비병·외주 정비·외주 검사·재작업)과 가동률을 만든다.
# ---------------------------------------------------------------------------
UNITS = ["가상A부대", "가상B부대", "가상C부대", "가상D부대", "가상E부대", "가상F부대"]
ETYPES = ["유형A", "유형B", "유형C"]
WINDOW = 90                                       # 관측 기간(일)
CAUSES = ["부품 대기", "정비병 부족", "외주 정비", "검사·재작업"]
CAUSE_COL = {"부품 대기": "parts_wait_days", "정비병 부족": "labor_wait_days", "외주 정비": "outsource_days", "검사·재작업": "inspect_wait_days"}
NUM_FIELDS = ["age_years", "op_hours", "maint_orders", "parts_wait_days", "labor_wait_days", "outsource_days", "inspect_wait_days", "rework_count",
              "spec_mismatch", "defect_reports", "parts_stock_pct", "mechanic_fill_pct", "avail_pct"]
FEAT = ["age_years", "maint_orders", "parts_stock_pct", "mechanic_fill_pct", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch", "defect_reports", "etype_B", "etype_C"]
FEAT_KO = {"age_years": "장비 연령", "maint_orders": "정비 요청 건수", "parts_stock_pct": "부품 재고 가용률", "mechanic_fill_pct": "정비병 충원율", "outsource_days": "외주 정비 일수",
           "inspect_wait_days": "외주 검사 대기 일수", "rework_count": "재작업 건수", "spec_mismatch": "규격·제원 불일치", "defect_reports": "하자·불만 접수",
           "etype_B": "유형B 여부", "etype_C": "유형C 여부"}


def unit_profile(seed=3):
    """부대별 고정 특성: 부품 재고 수준, 정비병 충원율, 외주 정비소 거리(일수 가중), 외주 검사 혼잡도."""
    r = np.random.default_rng(seed)
    return pd.DataFrame({"unit": UNITS, "stock": r.uniform(45, 95, len(UNITS)), "fill": r.uniform(55, 98, len(UNITS)),
                         "dist": r.uniform(0.8, 1.6, len(UNITS)), "queue": r.uniform(0.7, 2.2, len(UNITS))})


def make_fleet(n=300, seed=7, shock=0.0):
    """장비 n대의 90일 관측 자료(필드 16개)와 정답 요인을 만든다. shock은 외주 검사 병목 악화 정도(AAR용 환경 변화)."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    ui = rng.integers(0, len(UNITS), n)
    et = rng.choice(ETYPES, n, p=[.35, .3, .35])
    age = np.clip(rng.gamma(4, 2.2, n), 0.5, 30).round(1)
    eff = np.array([{"유형A": 1.15, "유형B": 1.0, "유형C": 0.9}[t] for t in et])
    orders = rng.poisson(np.clip(1.2 + 0.12 * age, 0.5, 6) * eff)
    stock = np.clip(pf["stock"].to_numpy()[ui] + rng.normal(0, 8, n), 20, 100).round(0)
    fill = np.clip(pf["fill"].to_numpy()[ui] + rng.normal(0, 6, n), 30, 100).round(0)
    parts_w = (orders * rng.gamma(2, 0.55 * (1 + ((100 - stock) / 28) ** 2) / 2)).round(0)
    labor_w = (orders * rng.gamma(2, 0.7 * (1 + ((100 - fill) / 24) ** 2) / 2)).round(0)
    n_out = rng.binomial(orders, 0.3)
    out_d = (n_out * rng.gamma(3, 4.5 * pf["dist"].to_numpy()[ui] / 3)).round(0)
    insp_w = (n_out * rng.gamma(2, (2.6 + shock) * pf["queue"].to_numpy()[ui] / 2)).round(0)
    spec = rng.poisson(0.2 + 0.04 * age)
    rework = rng.binomial(n_out, np.clip(0.08 + 0.1 * spec, 0, .6))
    defect = rng.poisson(0.3 + 0.7 * rework)
    base_rep = orders * 1.8
    down = base_rep + parts_w + labor_w + out_d + insp_w + rework * 4 + rng.normal(0, 1.2, n)
    comp = np.column_stack([parts_w, labor_w, out_d, insp_w + rework * 4])
    df = pd.DataFrame({"equip_id": [f"E-{i + 1:04d}" for i in range(n)], "unit": np.array(UNITS)[ui], "etype": et, "age_years": age,
                       "op_hours": np.clip(rng.normal(320 - 4 * age, 70, n), 40, 600).round(0), "maint_orders": orders,
                       "parts_wait_days": parts_w, "labor_wait_days": labor_w, "outsource_days": out_d, "inspect_wait_days": insp_w,
                       "rework_count": rework, "spec_mismatch": spec, "defect_reports": defect, "parts_stock_pct": stock, "mechanic_fill_pct": fill,
                       "avail_pct": (100 * (1 - np.clip(down, 0, WINDOW) / WINDOW)).round(1)})
    df["main_cause"] = np.where(comp.max(axis=1) < 3, "없음", np.array(CAUSES)[comp.argmax(axis=1)])
    return df


def feat_frame(df):
    """회귀·분석용 설명 변수 표(유형은 더미 변수)."""
    x = df.copy()
    x["etype_B"], x["etype_C"] = (x["etype"] == "유형B").astype(int), (x["etype"] == "유형C").astype(int)
    return x[FEAT]


def make_trend(months=12, seed=5, shock_unit=None):
    """[교육용가상자료] 부대 x 유형 x 월 평균 가동률 추세. shock_unit 부대는 하반기에 외주 검사 병목으로 가동률이 떨어진다."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    rows = []
    for _, u in pf.iterrows():
        base = 62 + 0.22 * u["stock"] + 0.12 * u["fill"] - 3.0 * u["queue"]
        for t in ETYPES:
            off = {"유형A": -3.0, "유형B": 0.0, "유형C": 2.0}[t]
            for m in range(months):
                v = base + off + 2.0 * np.sin(m / 12 * 2 * np.pi) + rng.normal(0, 1.0)
                if shock_unit == u["unit"] and m >= months // 2:
                    v -= 1.5 * (m - months // 2 + 1)
                rows.append((u["unit"], t, m + 1, round(float(np.clip(v, 30, 99)), 1)))
    return pd.DataFrame(rows, columns=["unit", "etype", "month", "avail_pct"])

# ---- 가동률 회귀 모델과 저가동 요인 분류·추천 모델(외부 AI 서비스 없이 scikit-learn 사용) ----
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

CAUSE_COLS = ["age_years", "maint_orders", "parts_stock_pct", "mechanic_fill_pct", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch",
              "defect_reports", "parts_wait_days", "labor_wait_days"]
ACTION = {"부품 대기": "부품 사전 청구·재고 보강", "정비병 부족": "정비병 재배치·교육 일정 조정", "외주 정비": "외주 정비 일정 조율·내부 정비 확대", "검사·재작업": "검사 일정 조율·규격 사전 대조"}


def cause_frame(df):
    x = df[CAUSE_COLS].copy()
    x["out_insp"] = x["outsource_days"] + x["inspect_wait_days"] + 4 * x["rework_count"]
    x["wait_pm"] = (x["parts_wait_days"] + x["labor_wait_days"]) / (x["maint_orders"] + 1)
    return x


def fit_avail(df, alpha=1.0):
    """가동률(%) 회귀: 표준화 + 릿지 회귀."""
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(feat_frame(df), df["avail_pct"])


def fit_cause(df, cut=85, C=5):
    """가동률이 cut% 미만인 장비의 주요 저가동 요인(4종)을 분류한다."""
    low = df[(df["avail_pct"] < cut) & (df["main_cause"] != "없음")]
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000)).fit(cause_frame(low), low["main_cause"])


SIM_FEAT = ["age_years", "maint_orders", "spec_mismatch", "stock_gap_x_orders", "fill_gap_x_orders", "parts_stock_pct", "mechanic_fill_pct", "etype_B", "etype_C"]


def sim_frame(df):
    """자원 재배치 시뮬레이션용 설명 변수: 부족분이 클수록 대기가 더 빠르게 늘고 정비 요청 건수와 곱해져 커진다는 구조를 제곱·상호작용 항으로 둔다."""
    x = df.copy()
    x["etype_B"], x["etype_C"] = (x["etype"] == "유형B").astype(int), (x["etype"] == "유형C").astype(int)
    x["stock_gap_x_orders"] = x["maint_orders"] * ((100 - x["parts_stock_pct"]) / 28) ** 2
    x["fill_gap_x_orders"] = x["maint_orders"] * ((100 - x["mechanic_fill_pct"]) / 24) ** 2
    return x[SIM_FEAT]


def fit_sim(df, alpha=0.5):
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(sim_frame(df), df["avail_pct"])

# ---- 평시 자료로 학습한 기존 예측 모델과, 외주 검사 병목이 악화된 새 분기(실제) ----
hist = make_fleet(900, seed=21)
m0 = fit_sim(hist)
now = make_fleet(1800, seed=51, shock=4.0)                       # 실제: 외주 검사 대기 병목 악화(shock) -> 기존 모델은 모름
aar, test = now.iloc[:600], now.iloc[600:]                       # AAR 표본(분기 초 실측) / 이후 검증 자료
CELL = ["unit", "etype"]


def cells(df, pred):
    t = df[CELL].copy(); t["actual"], t["pred"] = df["avail_pct"].to_numpy(), pred
    g = t.groupby(CELL).mean(); g["err"] = g["actual"] - g["pred"]; return g


def score(df, pred, tol=3.0):
    """
    [TODO : 학생 실습 9] AAR 예측치와 실제치 편차를 평가하여 허용오차(tol=%p) 이내 적중률(hit), MAE, 편향(bias)을 계산하세요.
    - 힌트 1: cells(df, pred) 함수를 호출하여 부대 x 장비유형별 평균 오차(err = actual - pred) 데이터프레임을 구합니다.
    - 힌트 2: hit은 오차 절댓값이 tol(3.0%p) 이하인 셀의 비율((c['err'].abs() <= tol).mean())입니다.
    - 힌트 3: 전체 레코드 기준 MAE(평균절대오차)와 bias(평균편향)를 계산합니다.
    - 반환값: {"hit": float, "mae": float, "bias": float, "cells": DataFrame}
    """
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.


# 1) AAR: 예측치와 실제치의 편차
p_aar = m0.predict(sim_frame(aar))
s0 = score(aar, p_aar)
print(f"[AAR 편차 분석] 예측 대비 실제: 평균 편향 {s0['bias']:+.2f}%p(실제가 낮음), 개별 MAE {s0['mae']:.2f}%p, 셀 18곳 중 오차 3%p 이내 {s0['hit']:.0%}")
worst = s0["cells"].sort_values("err").head(3)
print("오차가 큰 셀:", ", ".join(f"{i[0]}/{i[1]} {r.err:+.1f}%p" for i, r in worst.iterrows()))
comp = {"부품 대기": "parts_wait_days", "정비병 부족": "labor_wait_days", "외주 정비": "outsource_days", "외주 검사 대기": "inspect_wait_days", "재작업": "rework_count"}
contrib = {k: -(aar[c].mean() - hist[c].mean()) / WINDOW * 100 * (4 if c == "rework_count" else 1) for k, c in comp.items()}
print("오차 원인 분해(평시 대비 변화가 가동률에 미친 영향 %p):", ", ".join(f"{k} {v:+.2f}" for k, v in contrib.items()))
cause = min(contrib, key=contrib.get)
print(f"-> 가장 큰 원인: {cause} (예측 모델의 입력에 검사 병목 정보가 없었음). 재발 방지: 병목 변수 추가와 최근 자료 반영")

# 2) 재보정 패치 후보를 AAR 표본으로 만들고 검증 자료(test)로 평가
b_global = s0["bias"]
b_cell = s0["cells"]["err"]
unit_insp = aar.groupby("unit")["inspect_wait_days"].mean()            # P2: 부대별 최근 외주 검사 대기(병목 변수)


def sim2(df):
    x = sim_frame(df).copy(); x["insp_x_orders"] = df["unit"].map(unit_insp).to_numpy() * df["maint_orders"].to_numpy(); return x


def fit2(alpha, w):
    tr = pd.concat([hist.assign(_w=1.0), aar.assign(_w=w)]); x = sim2(tr)
    sc = StandardScaler().fit(x); return sc, Ridge(alpha=alpha).fit(sc.transform(x), tr["avail_pct"], sample_weight=tr["_w"])


for_cv = {}
for a in (0.1, 0.5, 2.0, 10.0, 50.0):                                  # P4: 정규화 강도를 AAR 표본 5분할 교차검증으로 선택
    e = []
    for tri, vai in KFold(5, shuffle=True, random_state=0).split(aar):
        tr = pd.concat([hist.assign(_w=1.0), aar.iloc[tri].assign(_w=3.0)]); x = sim2(tr); sc = StandardScaler().fit(x)
        r = Ridge(alpha=a).fit(sc.transform(x), tr["avail_pct"], sample_weight=tr["_w"])
        e.append(np.mean(np.abs(r.predict(sc.transform(sim2(aar.iloc[vai]))) - aar.iloc[vai]["avail_pct"])))
    for_cv[a] = float(np.mean(e))
alpha = min(for_cv, key=for_cv.get)
sc3, r3 = fit2(0.5, 3.0); sc4, r4 = fit2(alpha, 3.0)
cand = {"원 모델": m0.predict(sim_frame(test)),
        "P1 전체 편향 보정": m0.predict(sim_frame(test)) + b_global,
        "P1' 셀별 편향 보정": m0.predict(sim_frame(test)) + test[CELL].apply(tuple, axis=1).map(b_cell.to_dict()).to_numpy(),
        "P2+P3 병목 변수+최근 자료 가중": r3.predict(sc3.transform(sim2(test))),
        f"P4 +정규화 강도 {alpha}": r4.predict(sc4.transform(sim2(test)))}
print("\n[재보정 패치 검증(검증 자료 1,200대, 셀 18곳)]")
res = {}
for k, p in cand.items():
    res[k] = score(test, p); print(f"  {k:26s} 셀 적중 {res[k]['hit']:4.0%}  개별 MAE {res[k]['mae']:5.2f}  편향 {res[k]['bias']:+5.2f}")
best = min((k for k in cand if k != "원 모델"), key=lambda k: (-res[k]["hit"], res[k]["mae"]))
b = res[best]; o = res["원 모델"]
print(f"채택: {best} -> 셀 적중 {o['hit']:.0%} -> {b['hit']:.0%} ({(b['hit'] - o['hit']) * 100:+.0f}%p, 목표 +10%p 이상: {'달성' if b['hit'] - o['hit'] >= 0.10 else '미달'}), MAE {o['mae']:.2f} -> {b['mae']:.2f}")

# 3) P5 예측 구간(잔차 90% 분위수)과 재발 오류
pf = cand[best]; resid_aar = aar["avail_pct"].to_numpy() - r4.predict(sc4.transform(sim2(aar)))
q = float(np.quantile(np.abs(resid_aar), 0.9))
cover = float(np.mean(np.abs(test["avail_pct"].to_numpy() - pf) <= q))
bad0 = set(i for i, r in s0["cells"].iterrows() if abs(r.err) > 3)
recur = [i for i in bad0 if abs(b["cells"].loc[i, "err"]) > 3]
print("장비 1대 단위 예측은 외주 검사·재작업 같은 개별 사건을 알 수 없어 오차가 크므로, 부대·유형 셀 평균으로 계획에 쓴다.")
print(f"P5 예측 구간 ±{q:.1f}%p: 검증 포괄률 {cover:.0%}(목표 약 90%). 재보정 전 오차 3%p 초과 셀 {len(bad0)}곳 중 재발 {len(recur)}곳 {recur[:3] if recur else ''}")

# 4) UAT 체크리스트(측정값 기반 가상 점검) 10항목 x 10점
uat = {"셀 적중률 80% 이상": b["hit"] >= 0.8, "개별 MAE 12%p 이하(장비 단위는 참고용)": b["mae"] <= 12, "평균 편향 절대값 1.0%p 이하": abs(b["bias"]) <= 1.0, "예측 구간 포괄률 85~95%": 0.85 <= cover <= 0.95,
       "재발 오류 0건": len(recur) == 0, "원인 분해표 제공": len(contrib) == 5, "재보정 전후 비교표 제공": len(cand) == 5, "파라미터 YAML 출력": True,
       "이전 파라미터로 되돌리기 가능(원 모델 보존)": m0 is not None, "사람 확인 문구 포함(자동 적용 금지)": True}
pts = sum(10 for v in uat.values() if v)
print(f"\n[UAT 체크리스트] {pts}점/100점 (목표 90점 이상: {'달성' if pts >= 90 else '미달'}), 미충족: {[k for k, v in uat.items() if not v] or '없음'}")
print("개선 요구 반영 5건: P1 편향 보정, P2 외주 검사 병목 변수 추가, P3 최근 자료 가중(x3), P4 정규화 강도 교차검증 선택, P5 예측 구간 표시 -> 모두 반영")

YAML = f"""[tuned_params 제안: 담당자 확인 후 적용]
model: ridge
alpha: {alpha}
recent_weight: 3.0
extra_feature: insp_x_orders   # 부대별 최근 외주 검사 대기 x 정비 요청 건수
bias_cells: {len(b_cell)}
interval_pct90: {q:.1f}
retrain_trigger: 셀 적중률 70% 미만 또는 편향 절대값 1.0%p 초과"""
print("\n" + YAML)
print("지속 학습 연계: 분기마다 AAR 표본을 수집해 위 절차를 반복하되, 재보정 적용은 정비관·군수참모 확인 후에 한다. 가상 자료 기준 결과이다.")

plt.figure(figsize=(8, 4))
ks = list(res); plt.bar([f"S{i}" for i in range(len(ks))], [res[k]["hit"] * 100 for k in ks], color=["tab:gray"] + ["tab:blue"] * (len(ks) - 1))
plt.ylabel("cells within 3%p (%)"); plt.title("Forecast hit rate: base (S0) vs recalibration patches (S1-S4)"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 09 예상 실행 결과 (Output)</b></summary>

```text
[AAR 편차 분석] 예측 대비 실제: 평균 편향 -4.60%p(실제가 낮음), 개별 MAE 11.44%p, 셀 18곳 중 오차 3%p 이내 33%
오차가 큰 셀: 가상D부대/유형A -11.0%p, 가상D부대/유형C -10.3%p, 가상A부대/유형B -9.1%p
오차 원인 분해(평시 대비 변화가 가동률에 미친 영향 %p): 부품 대기 -0.21, 정비병 부족 -0.45, 외주 정비 -0.01, 외주 검사 대기 -4.92, 재작업 +0.18
-> 가장 큰 원인: 외주 검사 대기 (예측 모델의 입력에 검사 병목 정보가 없었음). 재발 방지: 병목 변수 추가와 최근 자료 반영

[재보정 패치 검증(검증 자료 1,200대, 셀 18곳)]
  원 모델                       셀 적중  39%  개별 MAE 10.53  편향 -3.82
  P1 전체 편향 보정                셀 적중  78%  개별 MAE 11.48  편향 +0.79
  P1' 셀별 편향 보정               셀 적중  50%  개별 MAE 11.81  편향 +0.95
  P2+P3 병목 변수+최근 자료 가중       셀 적중  83%  개별 MAE 10.62  편향 -0.78
  P4 +정규화 강도 10.0            셀 적중  83%  개별 MAE 10.62  편향 -0.78
채택: P2+P3 병목 변수+최근 자료 가중 -> 셀 적중 39% -> 83% (+44%p, 목표 +10%p 이상: 달성), MAE 10.53 -> 10.62
장비 1대 단위 예측은 외주 검사·재작업 같은 개별 사건을 알 수 없어 오차가 크므로, 부대·유형 셀 평균으로 계획에 쓴다.
P5 예측 구간 ±25.2%p: 검증 포괄률 90%(목표 약 90%). 재보정 전 오차 3%p 초과 셀 12곳 중 재발 1곳 [('가상A부대', '유형B')]

[UAT 체크리스트] 90점/100점 (목표 90점 이상: 달성), 미충족: ['재발 오류 0건']
개선 요구 반영 5건: P1 편향 보정, P2 외주 검사 병목 변수 추가, P3 최근 자료 가중(x3), P4 정규화 강도 교차검증 선택, P5 예측 구간 표시 -> 모두 반영

[tuned_params 제안: 담당자 확인 후 적용]
model: ridge
alpha: 10.0
recent_weight: 3.0
extra_feature: insp_x_orders   # 부대별 최근 외주 검사 대기 x 정비 요청 건수
bias_cells: 18
interval_pct90: 25.2
retrain_trigger: 셀 적중률 70% 미만 또는 편향 절대값 1.0%p 초과
지속 학습 연계: 분기마다 AAR 표본을 수집해 위 절차를 반복하되, 재보정 적용은 정비관·군수참모 확인 후에 한다. 가상 자료 기준 결과이다.
```
</details>
